In [12]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
pd.set_option('display.max_columns',500)

In [3]:
df=pd.read_csv("cleaned_ai_job_market.csv")

In [4]:
df

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,...,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,skills_list
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7,Master's,239000.0,155000,290000,Boston,...,96,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),"['APIs', 'Planning Systems', 'Python', 'Cloud'..."
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2,Bachelor's,166000.0,90000,200000,London,...,82,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),"['Python', 'Documentation', 'LLM APIs', 'Promp..."
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4,Associate's,360000.0,160000,300000,Seattle,...,98,42.7,7.7,2026,1,1,1,1,Elite (>$300k),"['Vector DBs', 'Python', 'Prompt Engineering',..."
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3,Bachelor's,161000.0,130000,220000,Singapore,...,88,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),"['Feature Stores', 'Spark', 'ETL', 'Airflow', ..."
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,...,85,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),"['Data Analysis', 'Stakeholder Mgmt', 'Agile',..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,AIJOB1496,Multimodal AI Engineer,AI Engineering,Lead (10+ yrs),8,Master's,262000.0,160000,300000,Toronto,...,90,23.6,7.5,2026,1,1,1,0,Senior ($200-300k),"['Python', 'Vision-Language Models', 'Fine-tun..."
1496,AIJOB1497,NLP Engineer,AI Engineering,Senior (6-9 yrs),9,Bootcamp/Self-taught,163000.0,145000,290000,Berlin,...,91,66.5,7.8,2025,3,1,0,0,Upper-Mid ($150-200k),"['Transformers', 'Text Processing', 'LLMs', 'H..."
1497,AIJOB1498,AI Compliance Manager,Governance,Mid (3-5 yrs),10,Master's,127000.0,100000,200000,Zurich,...,68,15.3,8.3,2026,3,0,1,0,Mid ($100-150k),"['Documentation', 'Risk Management', 'Legal Kn..."
1498,AIJOB1499,NLP Engineer,AI Engineering,Lead (10+ yrs),4,Bachelor's,145000.0,145000,290000,Bangalore,...,91,78.8,7.4,2025,5,1,1,0,Mid ($100-150k),"['LLMs', 'Hugging Face', 'Python', 'Transforme..."


In [5]:
df['years_of_experience'].describe()

count    1500.000000
mean        6.216000
std         2.675216
min         1.000000
25%         4.000000
50%         6.000000
75%         8.000000
max        15.000000
Name: years_of_experience, dtype: float64

chi_square_test

In [6]:
table=pd.crosstab(df['experience_level'],df['job_category'])
print(table)

job_category      AI Engineering  Architecture  Business  Data Engineering  \
experience_level                                                             
Entry (0-2 yrs)              183            20        21                17   
Lead (10+ yrs)               194            12        12                13   
Mid (3-5 yrs)                174            12        16                 8   
Senior (6-9 yrs)             185             8        13                13   

job_category      Data Science  Governance  Infrastructure  ML Operations  \
experience_level                                                            
Entry (0-2 yrs)             29          33               8             10   
Lead (10+ yrs)              33          25              14             11   
Mid (3-5 yrs)               31          30              18             16   
Senior (6-9 yrs)            34          34              15             14   

job_category      Product  Research  Robotics  Security  
experience

In [13]:
chi2,p,dof,expected = chi2_contingency(table)
print("chisquare value :",chi2)
print("p value is :",p)

chisquare value : 32.67727056651603
p value is : 0.48309454589730105


so there is a significant relationship between the job category and the experience level

In [15]:
table=pd.crosstab(df['country'],df['annual_salary_usd'])
chi2,p,dof,expected = chi2_contingency(table)
print("chisquare value :",chi2)
print("p value is :",p)

chisquare value : 3351.630984640915
p value is : 0.041089641820275004


so there is a significant relationship between the country and the annual salary

In [16]:
table=pd.crosstab(df['education_required'],df['annual_salary_usd'])
chi2,p,dof,expected = chi2_contingency(table)
print("chisquare value :",chi2)
print("p value is :",p)

chisquare value : 1054.1225827086123
p value is : 0.07071749819578481


There is no significant difference between education required and annual salary

In [20]:
from scipy.stats import f_oneway

entry = df[df['experience_level']=='Entry (0-2 yrs)']['annual_salary_usd']
mid = df[df['experience_level']=='Mid (3-5 yrs)']['annual_salary_usd']
senior = df[df['experience_level']=='Senior (6-9 yrs)']['annual_salary_usd']
lead = df[df['experience_level']=='Lead (10+ yrs)']['annual_salary_usd']

f_stat, p_val = f_oneway(entry, mid, senior, lead)

print("F-statistic:", f_stat)
print("p-value:", p_val)

F-statistic: 188.78401453287867
p-value: 8.226321008237093e-104


A one-way ANOVA showed that annual salary differs significantly across experience levels (F = 188.78, p < 0.001).

In [31]:
from sklearn.linear_model import LinearRegression

X = pd.get_dummies(df[['experience_level','remote_work']], drop_first=True)
y = df['annual_salary_usd']

model = LinearRegression()
model.fit(X, y)

print("Model trained successfully")

Model trained successfully


In [36]:
coeff = pd.Series(model.coef_, index=X.columns)
coeff = coeff.sort_values(ascending = False)
print(coeff)

experience_level_Lead (10+ yrs)      90259.365042
experience_level_Senior (6-9 yrs)    64298.407741
experience_level_Mid (3-5 yrs)       26255.007893
remote_work_On-site                   -198.799507
remote_work_Hybrid                   -5421.045692
dtype: float64


Regression analysis shows that experience level significantly affects salary. Lead roles earn approximately $90k more than entry-level roles, while senior and mid-level roles earn about $64k and $26k more respectively. Additionally, fully remote jobs tend to offer slightly higher salaries compared to hybrid or onsite roles.